# Train MobileNet Posture Classifier

This notebook trains a lightweight MobileNet posture classifier from the current
Roboflow folder-format export:

`data/labeled/posture_classification`

The source export has only a `train` folder, so the notebook first creates a
stratified grouped train/val/test split under:

`data/processed/posture_classification_mobilenet_split`


## Hyperparameter Choice

Default model is `mobilenet_v3_small` because this classifier will run after the
human detector, possibly once per detected person. The settings favor practical
CPU inference while still fine-tuning the full network:

- `img_size=224`
- `batch_size=32`
- 5 frozen-head epochs, then 35 full fine-tuning epochs
- AdamW with differential learning rates
- class-weighted cross-entropy with label smoothing
- mild crop-safe augmentation only: horizontal flip, small affine jitter,
  brightness/contrast jitter

If final recall is weak for `person_laying` or `person_sitting`, the next
controlled experiment should switch `MODEL_NAME` to `mobilenet_v3_large`, not
increase image size first.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import copy
import json
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

SOURCE_DATASET = PROJECT_ROOT / "data" / "labeled" / "posture_classification"
SOURCE_TRAIN = SOURCE_DATASET / "train"
PREPARED_DATASET = PROJECT_ROOT / "data" / "processed" / "posture_classification_mobilenet_split"
RUN_DIR = PROJECT_ROOT / "runs" / "posture_classification" / "mobilenet_v3_small_full_roboflow"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42
MODEL_NAME = "mobilenet_v3_small"  # change to "mobilenet_v3_large" for the next accuracy-focused experiment
IMG_SIZE = 224
BATCH_SIZE = 32
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 35
PATIENCE = 9
NUM_WORKERS = 0  # safest for Windows/Jupyter CPU

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads()))

device = torch.device("cpu")
print("Project root:", PROJECT_ROOT)
print("Source dataset:", SOURCE_DATASET)
print("Prepared dataset:", PREPARED_DATASET)
print("Run dir:", RUN_DIR)
print("Device:", device)

if not SOURCE_TRAIN.exists():
    raise FileNotFoundError(f"Missing source folder: {SOURCE_TRAIN}")


## Build Stratified Grouped Split


In [ ]:
def original_group_key(image_path: Path) -> str:
    # Roboflow variants look like name.rf.<hash>.jpg.
    return image_path.stem.split(".rf.", 1)[0]


def collect_rows(source_train: Path):
    rows = []
    for class_dir in sorted(p for p in source_train.iterdir() if p.is_dir()):
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() in IMAGE_EXTS:
                rows.append(
                    {
                        "path": image_path,
                        "class_name": class_dir.name,
                        "group": f"{class_dir.name}::{original_group_key(image_path)}",
                    }
                )
    return rows


def split_grouped_stratified(rows, train_ratio=0.70, val_ratio=0.20, seed=SEED):
    groups_by_class = defaultdict(lambda: defaultdict(list))
    for row in rows:
        groups_by_class[row["class_name"]][row["group"]].append(row)

    split_rows = {"train": [], "val": [], "test": []}
    rng = random.Random(seed)

    for class_name, groups in groups_by_class.items():
        group_items = list(groups.items())
        rng.shuffle(group_items)
        n = len(group_items)
        train_end = int(n * train_ratio)
        val_end = train_end + int(n * val_ratio)
        assignments = [
            ("train", group_items[:train_end]),
            ("val", group_items[train_end:val_end]),
            ("test", group_items[val_end:]),
        ]
        for split, items in assignments:
            for _, group_rows in items:
                split_rows[split].extend(group_rows)
    return split_rows


rows = collect_rows(SOURCE_TRAIN)
if not rows:
    raise RuntimeError(f"No images found under {SOURCE_TRAIN}")

split_rows = split_grouped_stratified(rows)

if PREPARED_DATASET.exists():
    shutil.rmtree(PREPARED_DATASET)
for split, split_items in split_rows.items():
    for row in split_items:
        dst_dir = PREPARED_DATASET / split / row["class_name"]
        dst_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(row["path"], dst_dir / row["path"].name)

for split in ["train", "val", "test"]:
    counts = Counter(row["class_name"] for row in split_rows[split])
    print(split, dict(counts), "total", sum(counts.values()))

classes = sorted({row["class_name"] for row in rows})
print("Classes:", classes)


## Datasets And Transforms


In [ ]:
train_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(
            degrees=7,
            translate=(0.04, 0.04),
            scale=(0.95, 1.05),
            shear=0,
        ),
        transforms.ColorJitter(brightness=0.18, contrast=0.18),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

eval_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_ds = datasets.ImageFolder(PREPARED_DATASET / "train", transform=train_tf)
val_ds = datasets.ImageFolder(PREPARED_DATASET / "val", transform=eval_tf)
test_ds = datasets.ImageFolder(PREPARED_DATASET / "test", transform=eval_tf)

class_names = train_ds.classes
class_to_idx = train_ds.class_to_idx
idx_to_class = {idx: name for name, idx in class_to_idx.items()}
print("class_to_idx:", class_to_idx)

train_counts = Counter(label for _, label in train_ds.samples)
class_weights = torch.tensor(
    [len(train_ds) / (len(class_names) * train_counts[i]) for i in range(len(class_names))],
    dtype=torch.float32,
)
print("train counts:", {idx_to_class[k]: v for k, v in train_counts.items()})
print("class weights:", {idx_to_class[i]: float(class_weights[i]) for i in range(len(class_names))})

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


## Visual Sanity Check


In [ ]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    return (tensor.cpu() * std + mean).clamp(0, 1)


images, labels = next(iter(train_loader))
fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, image, label in zip(axes.ravel(), images[:12], labels[:12]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(idx_to_class[int(label)], fontsize=9)
    ax.axis("off")
plt.tight_layout()


## Model


In [ ]:
def build_model(model_name: str, num_classes: int):
    if model_name == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_small(weights=weights)
    elif model_name == "mobilenet_v3_large":
        weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
        model = models.mobilenet_v3_large(weights=weights)
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model


model = build_model(MODEL_NAME, len(class_names)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.05)
print(model.__class__.__name__, "parameters:", sum(p.numel() for p in model.parameters()))


## Training Helpers


In [ ]:
def confusion_matrix_from_preds(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm


def metrics_from_confusion(cm):
    total = cm.sum()
    accuracy = np.trace(cm) / total if total else 0.0
    per_class = []
    f1s = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        f1s.append(f1)
        per_class.append({"precision": precision, "recall": recall, "f1": f1})
    return accuracy, float(np.mean(f1s)), per_class


def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    y_true = []
    y_pred = []

    with torch.set_grad_enabled(train_mode):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if train_mode:
                loss.backward()
                optimizer.step()

            total_loss += float(loss.item()) * images.size(0)
            y_true.extend(labels.detach().cpu().numpy().tolist())
            y_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    cm = confusion_matrix_from_preds(y_true, y_pred, len(class_names))
    accuracy, macro_f1, per_class = metrics_from_confusion(cm)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "per_class": per_class,
        "confusion": cm,
    }


def set_features_trainable(model, trainable: bool):
    for param in model.features.parameters():
        param.requires_grad = trainable


## Train


In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)

history = []
best_state = None
best_score = -1.0
bad_epochs = 0

# Stage 1: train classifier head only.
set_features_trainable(model, False)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3,
    weight_decay=1e-4,
)

for epoch in range(1, HEAD_EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer)
    val_metrics = run_epoch(model, val_loader)
    history.append({"stage": "head", "epoch": epoch, "train": train_metrics, "val": val_metrics})
    print(
        f"head {epoch:02d}/{HEAD_EPOCHS} "
        f"train_loss={train_metrics['loss']:.4f} val_loss={val_metrics['loss']:.4f} "
        f"val_acc={val_metrics['accuracy']:.3f} val_macro_f1={val_metrics['macro_f1']:.3f}"
    )

# Stage 2: fine-tune all layers with lower backbone LR.
set_features_trainable(model, True)
optimizer = torch.optim.AdamW(
    [
        {"params": model.features.parameters(), "lr": 3e-5},
        {"params": model.classifier.parameters(), "lr": 3e-4},
    ],
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINE_TUNE_EPOCHS)

for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer)
    val_metrics = run_epoch(model, val_loader)
    scheduler.step()

    score = val_metrics["macro_f1"]
    history.append({"stage": "fine", "epoch": epoch, "train": train_metrics, "val": val_metrics})
    print(
        f"fine {epoch:02d}/{FINE_TUNE_EPOCHS} "
        f"train_loss={train_metrics['loss']:.4f} val_loss={val_metrics['loss']:.4f} "
        f"val_acc={val_metrics['accuracy']:.3f} val_macro_f1={val_metrics['macro_f1']:.3f}"
    )

    if score > best_score:
        best_score = score
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
        torch.save(
            {
                "model_name": MODEL_NAME,
                "state_dict": best_state,
                "class_to_idx": class_to_idx,
                "img_size": IMG_SIZE,
                "best_val_macro_f1": best_score,
            },
            RUN_DIR / "best.pt",
        )
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping after {epoch} fine-tune epochs.")
            break

print("Best val macro F1:", best_score)


## Test Best Checkpoint


In [ ]:
checkpoint = torch.load(RUN_DIR / "best.pt", map_location=device)
model = build_model(checkpoint["model_name"], len(class_names)).to(device)
model.load_state_dict(checkpoint["state_dict"])

test_metrics = run_epoch(model, test_loader)
print("test loss:", test_metrics["loss"])
print("test accuracy:", test_metrics["accuracy"])
print("test macro_f1:", test_metrics["macro_f1"])
print("confusion rows=true cols=pred")
print(class_names)
print(test_metrics["confusion"])
for i, row in enumerate(test_metrics["per_class"]):
    print(class_names[i], row)

(RUN_DIR / "class_to_idx.json").write_text(json.dumps(class_to_idx, indent=2), encoding="utf-8")


## Training Curves


In [ ]:
fine_rows = [row for row in history if row["stage"] == "fine"]
epochs = list(range(1, len(fine_rows) + 1))
train_loss = [row["train"]["loss"] for row in fine_rows]
val_loss = [row["val"]["loss"] for row in fine_rows]
val_f1 = [row["val"]["macro_f1"] for row in fine_rows]
val_acc = [row["val"]["accuracy"] for row in fine_rows]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_loss, label="train loss")
axes[0].plot(epochs, val_loss, label="val loss")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(epochs, val_acc, label="val accuracy")
axes[1].plot(epochs, val_f1, label="val macro F1")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()


## CPU Inference Timing


In [ ]:
def time_model(model, loader, repeats=5):
    model.eval()
    images, _ = next(iter(loader))
    images = images[:1].to(device)
    with torch.no_grad():
        model(images)
        start = time.perf_counter()
        for _ in range(repeats):
            model(images)
        elapsed = time.perf_counter() - start
    return elapsed * 1000 / repeats


ms = time_model(model, test_loader, repeats=30)
print(f"Single crop inference: {ms:.2f} ms/image on {device}")


## Saved-Model Evaluation Charts

Run these cells after training finishes. They reload the saved `best.pt`, evaluate the test split again, save CSV summaries, and render charts for confusion, per-class quality, confidence, and mistakes.


In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PREPARED_DATASET = PROJECT_ROOT / "data" / "processed" / "posture_classification_mobilenet_split"
RUN_DIR = PROJECT_ROOT / "runs" / "posture_classification" / "mobilenet_v3_small_full_roboflow"
BEST_WEIGHTS = RUN_DIR / "best.pt"
CLASS_MAP = RUN_DIR / "class_to_idx.json"
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f"Missing trained checkpoint: {BEST_WEIGHTS}")
if not (PREPARED_DATASET / "test").exists():
    raise FileNotFoundError(f"Missing test split: {PREPARED_DATASET / 'test'}")

device = torch.device("cpu")

def build_eval_model(model_name: str, num_classes: int):
    if model_name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=None)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = torch.nn.Linear(in_features, num_classes)
    elif model_name == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=None)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = torch.nn.Linear(in_features, num_classes)
    else:
        raise ValueError(f"Unsupported checkpoint model_name: {model_name}")
    return model

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_ds = datasets.ImageFolder(PREPARED_DATASET / "test", transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
class_names = test_ds.classes

try:
    checkpoint = torch.load(BEST_WEIGHTS, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(BEST_WEIGHTS, map_location=device)

model_name = checkpoint.get("model_name", "mobilenet_v3_small")
model = build_eval_model(model_name, len(class_names)).to(device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

records = []
criterion = torch.nn.CrossEntropyLoss(reduction="sum")
total_loss = 0.0

with torch.inference_mode():
    sample_offset = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item()
        probs = torch.softmax(logits, dim=1)
        conf, preds = probs.max(dim=1)

        for local_idx in range(labels.shape[0]):
            sample_path, _ = test_ds.samples[sample_offset + local_idx]
            true_idx = int(labels[local_idx].cpu())
            pred_idx = int(preds[local_idx].cpu())
            row = {
                "path": str(sample_path),
                "file": Path(sample_path).name,
                "true": class_names[true_idx],
                "pred": class_names[pred_idx],
                "true_idx": true_idx,
                "pred_idx": pred_idx,
                "confidence": float(conf[local_idx].cpu()),
                "correct": true_idx == pred_idx,
            }
            for idx, name in enumerate(class_names):
                row[f"prob_{name}"] = float(probs[local_idx, idx].cpu())
            records.append(row)
        sample_offset += labels.shape[0]

pred_df = pd.DataFrame(records)
pred_df.to_csv(RUN_DIR / "test_predictions.csv", index=False)

print("checkpoint:", BEST_WEIGHTS)
print("model:", model_name)
print("test images:", len(pred_df))
print("test loss:", total_loss / max(1, len(pred_df)))
print("test accuracy:", pred_df["correct"].mean())
pred_df.head()


In [ ]:
num_classes = len(class_names)
cm = np.zeros((num_classes, num_classes), dtype=int)
for true_idx, pred_idx in zip(pred_df["true_idx"], pred_df["pred_idx"]):
    cm[int(true_idx), int(pred_idx)] += 1

rows = []
for idx, name in enumerate(class_names):
    tp = cm[idx, idx]
    fp = cm[:, idx].sum() - tp
    fn = cm[idx, :].sum() - tp
    support = cm[idx, :].sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    rows.append({
        "class": name,
        "support": int(support),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "errors": int(support - tp),
    })

metrics_df = pd.DataFrame(rows)
summary_df = pd.DataFrame([{
    "accuracy": float(np.trace(cm) / cm.sum()),
    "macro_precision": float(metrics_df["precision"].mean()),
    "macro_recall": float(metrics_df["recall"].mean()),
    "macro_f1": float(metrics_df["f1"].mean()),
    "test_images": int(cm.sum()),
    "errors": int((~pred_df["correct"]).sum()),
}])

metrics_df.to_csv(RUN_DIR / "test_per_class_metrics.csv", index=False)
summary_df.to_csv(RUN_DIR / "test_summary_metrics.csv", index=False)

print("Summary")
display(summary_df)
print("Per class")
display(metrics_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Confusion matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_xticks(range(num_classes), class_names, rotation=30, ha="right")
axes[0].set_yticks(range(num_classes), class_names)
for r in range(num_classes):
    for c in range(num_classes):
        axes[0].text(c, r, cm[r, c], ha="center", va="center", color="black")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

row_totals = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_totals, out=np.zeros_like(cm, dtype=float), where=row_totals != 0)
im = axes[1].imshow(cm_norm, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title("Recall-normalized confusion")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
axes[1].set_xticks(range(num_classes), class_names, rotation=30, ha="right")
axes[1].set_yticks(range(num_classes), class_names)
for r in range(num_classes):
    for c in range(num_classes):
        axes[1].text(c, r, f"{cm_norm[r, c]:.2f}", ha="center", va="center", color="black")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(RUN_DIR / "chart_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

metrics_df.set_index("class")[["precision", "recall", "f1"]].plot(kind="bar", ax=axes[0], ylim=(0, 1.05))
axes[0].set_title("Per-class precision, recall, F1")
axes[0].set_xlabel("")
axes[0].set_ylabel("score")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y", alpha=0.25)

colors = ["#2ca02c" if e == 0 else "#d62728" for e in metrics_df["errors"]]
axes[1].bar(metrics_df["class"], metrics_df["support"], color="#4c78a8", label="support")
axes[1].bar(metrics_df["class"], metrics_df["errors"], color=colors, label="errors")
axes[1].set_title("Test support and mistakes")
axes[1].set_xlabel("")
axes[1].set_ylabel("images")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.25)

fig.tight_layout()
fig.savefig(RUN_DIR / "chart_per_class_metrics.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
correct_conf = pred_df.loc[pred_df["correct"], "confidence"]
wrong_conf = pred_df.loc[~pred_df["correct"], "confidence"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(correct_conf, bins=12, alpha=0.8, label="correct", color="#2ca02c")
if len(wrong_conf):
    axes[0].hist(wrong_conf, bins=12, alpha=0.8, label="wrong", color="#d62728")
axes[0].set_title("Prediction confidence")
axes[0].set_xlabel("max class probability")
axes[0].set_ylabel("images")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.25)

error_pairs = (
    pred_df.loc[~pred_df["correct"]]
    .groupby(["true", "pred"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
if len(error_pairs):
    labels = error_pairs["true"] + " -> " + error_pairs["pred"]
    axes[1].bar(labels, error_pairs["count"], color="#d62728")
    axes[1].tick_params(axis="x", rotation=30)
else:
    axes[1].text(0.5, 0.5, "No mistakes", ha="center", va="center", transform=axes[1].transAxes)
axes[1].set_title("Mistake types")
axes[1].set_ylabel("images")
axes[1].grid(axis="y", alpha=0.25)

fig.tight_layout()
fig.savefig(RUN_DIR / "chart_confidence_and_errors.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
errors_df = pred_df.loc[~pred_df["correct"]].sort_values("confidence", ascending=False).reset_index(drop=True)
print("Mistakes:", len(errors_df))
display(errors_df[["file", "true", "pred", "confidence"]].head(20))

if len(errors_df):
    n = min(12, len(errors_df))
    cols = 4
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for i in range(n):
        row = errors_df.iloc[i]
        image = Image.open(row["path"]).convert("RGB")
        axes[i].imshow(image)
        axes[i].set_title(f"true: {row['true']}\npred: {row['pred']} ({row['confidence']:.2f})", fontsize=9)
    fig.tight_layout()
    fig.savefig(RUN_DIR / "chart_top_mistakes.png", dpi=160, bbox_inches="tight")
    plt.show()
